**`03_prepare_administrative_units`**

Prepare initial layer of administrative units with geometries and `AdminId` identifiers.

In [ ]:
%load_ext autoreload
%autoreload 2

# Administrative units identifiers
``openplaces`` organizes its data by ``admin_id`` (data class: ``AdminId``).

``admin_id`` is a geographical administrative index with hierarchical ``.levels`` of any depth.

- **0** - countries
- **1** - states/departments/...
- **2** - counties/municipalities/...
- **3** - subdivisions/towns/...
- **4** - neighborhoods/...
- **5** - ...

The initial built is derived from ISO and the Global Administrative Database (GADM).

In [ ]:
from openplaces.api import get_admin0, get_admin1, get_admin2
from openplaces.core.schema import AdminId
from openplaces.io.admin import get_admin0_iso, get_admin1_iso
from openplaces.io.ingest import ingest_recipe
from openplaces.recipe import get_recipe
from openplaces.timing import get_timer
from openplaces.utils import pretty_print

In [ ]:
# Overwrite existing outputs?
REDO = False

In [ ]:
# Start timer
timer = get_timer('prepare_administrative_units', verbose=True)

# `admin0`: countries / territories
The highest level of the administrative hierarchy.
## ISO countries
Top-level administrative identifiers, gap-filled to match GADM, ships with `openplaces`

In [ ]:
get_admin0_iso()

## GADM level 0
Example of how to download and ingest data from the Internet using a `recipe`:

In [ ]:
recipe = get_recipe(AdminId(), 'admin', source='admin0-gadm-4~1')
pretty_print(recipe)

In [ ]:
ingest_recipe(recipe, timer=timer, redo=REDO)

### Read result

In [ ]:
admin0 = get_admin0()
admin0.sample(5).T

# ``admin1``: states / departments

## ISO states

In [ ]:
admin1_iso = get_admin1_iso()
admin1_iso

## GADM level 1

In [ ]:
admin1_recipe = get_recipe(AdminId(), 'admin', source='admin1-gadm-4~1')
pretty_print(admin1_recipe)

In [ ]:
ingest_recipe(admin1_recipe, timer=timer, redo=REDO)

In [ ]:
admin1 = get_admin1()
admin1.sample(5).T

# ``admin2``: counties / municipalities

## GADM level 2

In [ ]:
admin2_recipe = get_recipe(AdminId(), 'admin', source='admin2-gadm-4~1')
pretty_print(admin2_recipe)

In [ ]:
ingest_recipe(admin2_recipe, timer=timer, redo=REDO)

In [ ]:
admin2 = get_admin2(
    columns=['name', 'type', 'admin1_name', 'admin0_name', 'admin2_id_gadm']
)
admin2

# ``admin3``: towns / ...

## GADM level 3
Unfinished. GADM only covers a fraction of the world at Level 3 (Dec 1, 2025)

In [ ]:
admin3_recipe = get_recipe(AdminId(), 'admin', source='admin3-gadm-4~1')
pretty_print(admin3_recipe)

In [ ]:
ingest_recipe(admin3_recipe, timer=timer, redo=REDO)

In [ ]:
import pandas as pd

from openplaces.path import cache_path

admin3_path = cache_path(
    admin3_recipe['admin_id'],
    admin3_recipe['entity'],
    filename=admin3_recipe['cache_filename'],
)
admin3 = pd.read_parquet(admin3_path)

admin3.sample(5).T

In [ ]:
admin3 = admin3.join(
    admin2.reset_index()
    .set_index('admin2_id_gadm')[['admin2_id', 'name']]
    .rename(columns={'name': 'admin2_name_test'}),
    on='admin2_id_gadm',
)
admin3.sample(5).T

In [ ]:
import geopandas as gpd

admin3_geo_path = cache_path(
    admin3_recipe['admin_id'],
    admin3_recipe['entity'],
    filename=admin3_recipe['cache_filename'] + '_geo',
)
admin3_geo = gpd.read_parquet(admin3_geo_path)
admin3_geo.plot()

# Wrap up

In [ ]:
timer.summary()
timer.save()